# Modèles non linéaires pour `/recommend`

| | |
|---|---|
| **Objectif** | voir si des modèles non linéaires font mieux que la meilleure régression linéaire du notebook 13, si ses variables leur servent encore, puis retenir trois finalistes pour le notebook 15 |
| **Entrées** | `data/processed/recommend_training_dataset.csv` (notebook 06) ; coordonnées des pays tirées de `data/geo/ne_110m_admin_0_countries.geojson` |
| **Sorties** | runs MLflow (comparaison, tuning léger, tuning approfondi) ; réglages des trois finalistes, reportés dans `RECOMMEND_FINALISTS` du code commun |
| **Cible** | `yield_t_ha`, rendement en t/ha ; métrique principale : **RMSE** |
| **Protocole** | validation temporelle 2008-2012, chaque année prédite par un modèle appris sur les années précédentes ; 2013, réservée au test final, est retirée dès le découpage |
| **Features** | cinq représentations, détaillées plus bas : `base`, `geography`, `crop_x_geography`, `history`, `geography_country` |

## Imports

In [1]:
import tempfile
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from threadpoolctl import threadpool_limits
from xgboost import XGBRegressor

from agritech.config import SEED
from agritech.evaluation import compare_cv_folds
from agritech.geo import charger_coordonnees
from agritech.modeling import RunInfo, experiment_pipeline, run_experiment
from agritech.notification import notify
from agritech.recommend_config import (
    RECOMMEND_COLUMNS,
    RECOMMEND_CROPS,
    RECOMMEND_DATASET,
    RECOMMEND_GEOGRAPHY,
    RECOMMEND_HISTORICAL_CONDITIONS,
    RECOMMEND_LOG_CONDITIONS,
    RECOMMEND_ROWS,
    RECOMMEND_TARGET,
    RECOMMEND_TEST_YEAR,
    RECOMMEND_VALIDATION_YEARS,
)
from agritech.recommend_features import CropInteractions, add_historical_conditions, add_recommend_features
from agritech.tracking import run_tags, setup_mlflow
from agritech.training_data import load_dataset, temporal_cv, temporal_protocol_params, temporal_split
from agritech.tuning import log_trials, position_in_range, random_search, value_effects

# Données et protocole

Mêmes données, mêmes variables et mêmes 5 folds que le notebook 13.

In [2]:
debut_notebook = time.perf_counter()  # durée totale, affichée à la fin
df = load_dataset(RECOMMEND_DATASET, RECOMMEND_ROWS, RECOMMEND_COLUMNS, non_negative=[RECOMMEND_TARGET])
df = add_historical_conditions(add_recommend_features(df, charger_coordonnees()))

VARIABLES = ["iso3", "crop", "year", *RECOMMEND_LOG_CONDITIONS, *RECOMMEND_GEOGRAPHY, "temp_hist", "log_pest_hist"]
X_train, X_test, y_train, y_test = temporal_split(df, VARIABLES, RECOMMEND_TARGET, RECOMMEND_TEST_YEAR)
PARAMS_PROTOCOLE = temporal_protocol_params(
    RECOMMEND_DATASET, RECOMMEND_TARGET, X_train, X_test, RECOMMEND_VALIDATION_YEARS, RECOMMEND_TEST_YEAR
)
del df, X_test, y_test  # 2013 n'est plus accessible dans la suite du notebook
assert X_train["year"].max() < RECOMMEND_TEST_YEAR

print()
ANNEES = RECOMMEND_VALIDATION_YEARS
cv = temporal_cv(X_train["year"], ANNEES, RECOMMEND_TEST_YEAR)

fichier            : data/processed/recommend_training_dataset.csv
lignes x colonnes  : (16319, 8)
valeurs manquantes : 0
valeurs < 0        : 0 (yield_t_ha)
total   : 16319 lignes, 1990-2013
X_train : (15624, 12), 1990-2012
X_test  : (695, 12), 2013, réservé à l'évaluation finale
y_train : (15624,)
y_test  : (695,)

validation 2008 : apprentissage 1990-2007 (12152 lignes) | 694 lignes évaluées
validation 2009 : apprentissage 1990-2008 (12846 lignes) | 692 lignes évaluées
validation 2010 : apprentissage 1990-2009 (13538 lignes) | 695 lignes évaluées
validation 2011 : apprentissage 1990-2010 (14233 lignes) | 695 lignes évaluées
validation 2012 : apprentissage 1990-2011 (14928 lignes) | 696 lignes évaluées


In [3]:
# un tag d'étape par phase, pour les distinguer dans MLflow
experience = setup_mlflow("recommend")
TAGS = {
    phase: run_tags(service="recommend", stage=phase, notebook="14_recommend_nonlinear_models.ipynb")
    for phase in ["comparison", "light_tuning", "deep_tuning"]
}

expérience MLflow : oc_p12_agritech_recommend


## Modèles

| Modèle | Réglages de départ | `crop` et `iso3` |
|---|---|---|
| `RandomForest`, `ExtraTrees` | 300 arbres, autres réglages par défaut | one-hot |
| `HistGradientBoosting` | 500 itérations, pas de 0,05 | catégories natives |
| `XGBoost` | 500 arbres, pas de 0,05, profondeur 6 | catégories natives |
| `LightGBM` | 500 arbres, pas de 0,05, 31 feuilles | catégories natives |
| `CatBoost` | par défaut : 1 000 itérations, profondeur 6, pas automatique | one-hot |

Pour les boostings, on compare les catégories natives et le one-hot, puis on garde le meilleur encodage.

In [4]:
limite_openmp = threadpool_limits(1, user_api="openmp")  # HistGradientBoosting sur un cœur, comme les autres boostings

# famille : (modèle de départ, catégories natives, réglages enregistrés dans MLflow)
MODELES = {
    "random_forest": (RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED), False,
                      {"n_estimators": 300, "max_features": 1.0, "min_samples_leaf": 1}),
    "extra_trees": (ExtraTreesRegressor(n_estimators=300, n_jobs=-1, random_state=SEED), False,
                    {"n_estimators": 300, "max_features": 1.0, "min_samples_leaf": 1}),
    "hist_gradient_boosting": (
        HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, early_stopping=False, random_state=SEED), True,
        {"max_iter": 500, "learning_rate": 0.05, "max_leaf_nodes": 31},
    ),
    "xgboost": (XGBRegressor(n_estimators=500, learning_rate=0.05, enable_categorical=True, n_jobs=1, random_state=SEED),
                True, {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 6}),
    "lightgbm": (LGBMRegressor(n_estimators=500, learning_rate=0.05, n_jobs=1, random_state=SEED, verbose=-1), True,
                 {"n_estimators": 500, "learning_rate": 0.05, "num_leaves": 31}),
    "catboost": (CatBoostRegressor(thread_count=8, verbose=0, allow_writing_files=False, random_seed=SEED), False,
                 {"iterations": 1000, "depth": 6, "learning_rate": "auto"}),
}

## Représentations

Une représentation est un jeu de variables, comme au notebook 13 :

| Nom | Variables | Lignes |
|---|---|---|
| `base` | `crop`, `year`, `avg_temp`, `rain_mm`, `log_pesticides` | 1990-2012 |
| `geography` | `base` + `lat_abs`, `geo_x`, `geo_y`, `geo_z` | 1990-2012 |
| `crop_x_geography` | `geography` + interactions avec `crop` : une colonne par culture pour chaque condition et variable géographique (`avg_temp`, `rain_mm`, `log_pesticides`, `lat_abs`, `geo_x`, `geo_y`, `geo_z`) ; c'est la meilleure régression linéaire du notebook 13 | 1990-2012 |
| `history` | `crop`, `year`, `temp_hist`, `rain_mm`, `log_pest_hist`, `lat_abs`, `geo_x`, `geo_y`, `geo_z` | lignes avec historique, 1991-2012 |
| `geography_country` | `geography` + `iso3`, comparée à part | 1990-2012 |

`history` remplace `avg_temp` et `log_pesticides` par `temp_hist` et `log_pest_hist` : les conditions de l'année à
prédire ne sont pas encore connues au moment de la recommandation, on utilise donc les moyennes des 3 années
précédentes, disponibles à ce moment-là (logarithme pour les pesticides). `rain_mm` ne change pas : c'est une valeur
fixe par pays.

`lat_abs` est la latitude absolue du pays, `geo_x`, `geo_y` et `geo_z` sa position sur le globe.

In [5]:
CONDITIONS = RECOMMEND_LOG_CONDITIONS  # avg_temp, rain_mm, log_pesticides
GEOGRAPHIE = RECOMMEND_GEOGRAPHY  # lat_abs, geo_x, geo_y, geo_z
BASE = ["year", *CONDITIONS]
BASE_GEOGRAPHIE = [*BASE, *GEOGRAPHIE]
HISTORIQUE = ["year", *RECOMMEND_HISTORICAL_CONDITIONS, *GEOGRAPHIE]  # temp_hist, rain_mm, log_pest_hist

# représentation : (catégorielles, numériques, colonnes ajoutées dans le pipeline)
REPRESENTATIONS = {
    "base": (["crop"], BASE, None),  # sans géographie
    "crop_x_geography": (["crop"], BASE_GEOGRAPHIE, CropInteractions(CONDITIONS + GEOGRAPHIE)),
    "geography": (["crop"], BASE_GEOGRAPHIE, None),
    "history": (["crop"], HISTORIQUE, None),
    "geography_country": (["crop", "iso3"], BASE_GEOGRAPHIE, None),
}

# conditions historiques : lignes qui ont un historique (pas la première année de chaque pays), mêmes années de validation
avec_historique = X_train["temp_hist"].notna().to_numpy()
X_hist, y_hist = X_train[avec_historique], y_train[avec_historique]
PARAMS_HISTORIQUE = PARAMS_PROTOCOLE | {"n_train": len(X_hist), "train_years": f"{X_hist['year'].min()}-{X_hist['year'].max()}"}
cv_hist = temporal_cv(X_hist["year"], ANNEES, RECOMMEND_TEST_YEAR, verbose=False)
MEMES_LIGNES = (X_hist, y_hist, cv_hist, PARAMS_HISTORIQUE)

## Fonctions du notebook

In [6]:
resultats = {}  # « famille | représentation » : scores par année de validation
REFERENCE = "linear_regression | crop_x_geography"  # régression linéaire de référence : la meilleure du notebook 13


def evaluer(famille, modele, natif, representation, phase="comparison", params=None, donnees=None, suffixe="", run=True):
    "Validation temporelle d'un modèle sur une représentation, avec son run MLflow ; renvoie le nom du résultat."
    categorielles, numeriques, ajout = REPRESENTATIONS[representation]
    X, y, folds, protocole = donnees or (X_train, y_train, cv, PARAMS_PROTOCOLE)
    info = RunInfo(f"{famille}_{representation}_{phase}", representation, TAGS[phase], protocole,
                   params or {}, replace=True) if run else None
    nom = f"{famille} | {representation}{suffixe}"
    resultats[nom], _ = run_experiment(modele, X, y, folds, categorielles, numeriques, ajout, info, verbose=False,
                                       scale_numeric=isinstance(modele, LinearRegression), native_categorical=natif)
    return nom


def tableau(noms):
    "RMSE, MAE, R², RMSE par année, puis écart moyen à la référence et années où l'on fait mieux qu'elle."
    table = compare_cv_folds({nom: resultats[nom] for nom in [REFERENCE, *noms]}, REFERENCE, ANNEES)
    colonnes = {"écart moyen": "écart à la référence", "folds améliorés": "années meilleures que la référence"}
    return table.rename(columns=colonnes)[["RMSE", "MAE", "R²", *ANNEES, *colonnes.values()]].round(3)

# Référence

La **régression linéaire de référence** est la meilleure du notebook 13 (`crop_x_geography`, RMSE 4,673). Tous les
modèles de ce notebook lui sont comparés, année par année. Elle est recalculée ici, sans nouveau run MLflow.

In [7]:
evaluer("linear_regression", LinearRegression(), False, "crop_x_geography", run=False)
assert round(resultats[REFERENCE]["rmse"].mean(), 3) == 4.673  # notebook 13
tableau([])[["RMSE", "MAE", "R²", *ANNEES]]

,RMSE,MAE,R²,2008,2009,2010,2011,2012
linear_regression | crop_x_geography,4.673,2.871,0.693,4.531,4.754,4.675,4.704,4.699


**Observations :**

- On retrouve le score du notebook 13 : 4,673 t/ha, assez stable d'une année à l'autre (4,53 à 4,75).

# Première comparaison des modèles

Chaque famille avec ses réglages de départ. Pour les boostings, on garde d'abord le meilleur encodage des catégories
sur `geography` ; chaque famille est ensuite comparée, avec cet encodage, sur `base`, `geography` et
`crop_x_geography`.

In [8]:
# geography pour chaque famille ; pour les boostings, les deux encodages des catégories
encodages = {}
for famille, (modele, natif, reglages) in MODELES.items():
    depart = evaluer(famille, clone(modele), natif, "geography", params=reglages)
    if famille in ["hist_gradient_boosting", "xgboost", "lightgbm", "catboost"]:
        autre = clone(modele).set_params(cat_features=("crop",)) if famille == "catboost" else clone(modele)
        variante = evaluer(f"{famille}_{'onehot' if natif else 'native'}", autre, not natif, "geography", params=reglages)
        # l'encodage de départ (tableau des modèles) est bien le meilleur : il est gardé pour toute la suite
        assert resultats[depart]["rmse"].mean() < resultats[variante]["rmse"].mean()
        natives, onehot = (depart, variante) if natif else (variante, depart)
        encodages[famille] = {"RMSE catégories natives": resultats[natives]["rmse"].mean(),
                              "RMSE one-hot": resultats[onehot]["rmse"].mean(),
                              "encodage retenu": "catégories natives" if natif else "one-hot"}

pd.DataFrame.from_dict(encodages, orient="index").round(3)

,RMSE catégories natives,RMSE one-hot,encodage retenu
hist_gradient_boosting,1.897,1.978,catégories natives
xgboost,1.751,1.835,catégories natives
lightgbm,1.916,1.960,catégories natives
catboost,1.958,1.884,one-hot


In [9]:
# base et crop_x_geography, avec le même modèle, les mêmes réglages et le même encodage que geography
for famille, (modele, natif, reglages) in MODELES.items():
    for representation in ["base", "crop_x_geography"]:
        evaluer(famille, clone(modele), natif, representation, params=reglages)

noms = [f"{famille} | {representation}" for famille in MODELES
        for representation in ["base", "geography", "crop_x_geography"]]
scores = tableau(noms)[["RMSE", "MAE", "R²", *ANNEES]]
# famille et features en colonnes simples : le tableau peut être trié sur n'importe quelle colonne
premiere = pd.DataFrame([nom.split(" | ") for nom in scores.index], columns=["famille", "features"]).join(
    scores.reset_index(drop=True))
premiere

,famille,features,RMSE,MAE,R²,2008,2009,2010,2011,2012
0,linear_regression,crop_x_geography,4.673,2.871,0.693,4.531,4.754,4.675,4.704,4.699
1,random_forest,base,2.098,1.002,0.938,1.879,1.931,2.352,2.288,2.038
2,random_forest,geography,1.648,0.821,0.962,1.453,1.605,1.606,1.872,1.703
3,random_forest,crop_x_geography,1.677,0.838,0.960,1.538,1.612,1.605,1.902,1.729
4,extra_trees,base,1.706,0.846,0.959,1.524,1.663,1.795,1.873,1.674
5,extra_trees,geography,1.515,0.743,0.968,1.334,1.467,1.443,1.690,1.640
6,extra_trees,crop_x_geography,1.593,0.773,0.964,1.428,1.575,1.482,1.794,1.684
7,hist_gradient_boosting,base,2.286,1.306,0.926,2.059,2.324,2.437,2.355,2.254
8,hist_gradient_boosting,geography,1.897,1.094,0.949,1.737,1.919,1.842,1.995,1.991
9,hist_gradient_boosting,crop_x_geography,1.841,1.038,0.952,1.655,1.886,1.786,1.945,1.934


**Observations :**

- Toutes les familles battent largement la régression linéaire de référence, les 5 années, même sans géographie :
  1,71 à 2,31 t/ha sur `base`, contre 4,67.
- La géographie améliore toutes les familles, les 5 années : de 0,19 t/ha (ExtraTrees) à 0,45 t/ha (RandomForest) par
  rapport à `base`.
- Les interactions par culture du notebook 13 n'apportent pas de gain régulier : elles dégradent les forêts et XGBoost, et n'améliorent que légèrement les autres boostings. La suite part donc de `geography`.
- Catégories natives meilleures pour HistGradientBoosting, XGBoost et LightGBM, one-hot pour CatBoost : ce sont les
  encodages gardés pour la suite.
- Avec leurs réglages de départ, ExtraTrees sur `geography` est le plus précis (1,515 t/ha).

# Ajout du pays et conditions historiques

On teste encore deux ajouts à `geography` : le pays, représenté par la variable catégorielle `iso3`, et les conditions historiques.

In [10]:
ecarts = {}
for famille, (modele, natif, reglages) in MODELES.items():
    geo = f"{famille} | geography"
    pays = evaluer(famille, clone(modele), natif, "geography_country", params=reglages)
    table = compare_cv_folds({nom: resultats[nom] for nom in [geo, pays]}, geo)
    # conditions historiques : même famille, mêmes lignes
    geo_memes_lignes = evaluer(famille, clone(modele), natif, "geography", donnees=MEMES_LIGNES, suffixe=" (mêmes lignes)", run=False)
    historique = evaluer(famille, clone(modele), natif, "history", params=reglages, donnees=MEMES_LIGNES)
    table = pd.concat([table, compare_cv_folds({nom: resultats[nom] for nom in [geo_memes_lignes, historique]}, geo_memes_lignes)])
    ecarts[famille] = table.drop(index=[geo, geo_memes_lignes]).rename(index=lambda nom: nom.split(" | ")[1])

pd.concat({famille: table["écart moyen"].round(3).astype(str) + " (" + table["folds améliorés"] + ")"
           for famille, table in ecarts.items()}, axis=1)

,random_forest,extra_trees,hist_gradient_boosting,xgboost,lightgbm,catboost
geography_country,-0.031 (5/5),-0.011 (3/5),-0.314 (5/5),-0.281 (5/5),-0.295 (5/5),0.057 (0/5)
history,-0.055 (4/5),-0.086 (5/5),-0.052 (4/5),-0.088 (4/5),-0.059 (5/5),-0.049 (4/5)


**Observations :**

- L'ajout du pays aide surtout les boostings en catégories natives (environ −0,3 t/ha) ; il change peu les forêts et
  dégrade légèrement CatBoost. Comme au notebook 13, `iso3` reste une comparaison à part : il revient à reconnaître
  chaque pays.
- Les conditions historiques améliorent toutes les familles, 4 ou 5 années sur 5 : `history`, sans `iso3`, devient la
  représentation principale.

# Tuning léger

Le tuning se fait en deux temps, sur `history` (lignes qui ont un historique, mêmes 5 folds). Tuning léger : 40
configurations tirées au hasard pour chacune des 6 familles, pour choisir 3 finalistes. Tuning approfondi : toutes
les combinaisons d'une grille recentrée, pour chacun des 3 finalistes. Un run MLflow par configuration. La synthèse
compare le meilleur essai du tuning léger au réglage de départ de la section précédente, et garde le meilleur des
deux.

In [11]:
# pendant le tuning, les configurations tournent en parallèle : un cœur par modèle (mêmes prédictions qu'avec plusieurs)
UN_COEUR = {"random_forest": {"n_jobs": 1}, "extra_trees": {"n_jobs": 1}, "catboost": {"thread_count": 1}}
# réglage fixé pour tout le tuning : LightGBM tire des lignes à chaque arbre, sinon `subsample` n'a pas d'effet
FIXES = {"lightgbm": {"subsample_freq": 1}}
ESPACES_LEGERS = {
    "xgboost": {"n_estimators": [500, 1000, 2000], "learning_rate": [0.02, 0.05, 0.1], "max_depth": [6, 8, 10, 12],
                "min_child_weight": [1, 3, 5], "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.6, 0.8, 1.0],
                "reg_lambda": [0.1, 1.0, 5.0]},
    "lightgbm": {"n_estimators": [500, 1000, 2000], "learning_rate": [0.02, 0.05, 0.1], "num_leaves": [31, 63, 127, 255],
                 "min_child_samples": [2, 5, 10, 20], "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.6, 0.8, 1.0],
                 "reg_lambda": [0.0, 1.0, 5.0]},
    "hist_gradient_boosting": {"learning_rate": [0.03, 0.05, 0.1, 0.2], "max_iter": [500, 1000, 2000],
                               "max_leaf_nodes": [31, 63, 127, 255], "min_samples_leaf": [2, 5, 10, 20],
                               "l2_regularization": [0.0, 0.1, 1.0], "max_features": [0.5, 0.8, 1.0]},
    "catboost": {"iterations": [1000, 2000, 3000], "learning_rate": [0.03, 0.06, 0.1, 0.2], "depth": [6, 8, 10],
                 "l2_leaf_reg": [1, 3, 10]},
    "extra_trees": {"n_estimators": [300, 500, 1000], "max_features": [0.3, 0.5, 0.7, 1.0], "min_samples_leaf": [1, 2, 3, 5],
                    "bootstrap": [False, True]},
    "random_forest": {"n_estimators": [300, 500], "max_features": [0.3, 0.5, 0.7, 1.0], "min_samples_leaf": [1, 2, 3, 5],
                      "max_depth": [None, 15, 25]},
}


def rechercher(famille, espace, n_essais, phase, fixes=None, representation="history", donnees=MEMES_LIGNES):
    "Recherche aléatoire, un run MLflow par configuration ; range les essais dans `recherches` et les renvoie."
    modele, natif, _ = MODELES[famille]
    fixes = FIXES.get(famille, {}) | (fixes or {})
    modele = clone(modele).set_params(**UN_COEUR.get(famille, {}), **fixes)
    categorielles, numeriques, ajout = REPRESENTATIONS[representation]
    X, y, folds, protocole = donnees
    X = X[categorielles + numeriques]
    pipeline = experiment_pipeline(modele, categorielles, numeriques, ajout, native_categorical=natif)
    essais, duree = random_search(pipeline, espace, X, y, folds, n_essais, fold_labels=ANNEES, n_jobs=12)
    run = RunInfo(f"{famille}_{representation}_{phase}", representation, TAGS[phase], protocole,
                  {"tuning_phase": phase} | fixes, replace=True)
    log_trials(essais, run, pipeline, X, categorielles, numeriques, native_categorical=natif)
    recherches[famille, representation, phase] = (essais, duree, espace)
    print(f"{famille:24s} {representation} : {n_essais} configurations en {duree / 60:4.1f} min"
          f" | meilleure RMSE {essais['cv_rmse_mean'].iloc[0]:.3f}")
    return essais


recherches = {}  # (famille, représentation, phase) : (essais, durée, grille)
for famille, espace in ESPACES_LEGERS.items():
    rechercher(famille, espace, 40, "light_tuning")

xgboost                  history : 40 configurations en  0.7 min | meilleure RMSE 1.503


lightgbm                 history : 40 configurations en  0.4 min | meilleure RMSE 1.483


hist_gradient_boosting   history : 40 configurations en  1.0 min | meilleure RMSE 1.497


catboost                 history : 40 configurations en  2.4 min | meilleure RMSE 1.458


extra_trees              history : 40 configurations en  1.2 min | meilleure RMSE 1.447


random_forest            history : 40 configurations en  1.6 min | meilleure RMSE 1.570


In [12]:
# réglage de départ sur history (section précédente) contre meilleur des 40 essais : mêmes lignes, mêmes folds
synthese_legere = {}
for (famille, _, phase), (essais, duree, _) in recherches.items():
    if phase != "light_tuning":
        continue
    depart, essai = resultats[f"{famille} | history"], essais.iloc[0]
    tuning_meilleur = essai["cv_rmse_mean"] < depart["rmse"].mean()
    if tuning_meilleur:
        scores = {"RMSE": essai["cv_rmse_mean"], "MAE": essai["cv_mae_mean"], "R²": essai["cv_r2_mean"],
                  **{annee: essai[f"rmse_{annee}"] for annee in ANNEES}}
    else:
        scores = {"RMSE": depart["rmse"].mean(), "MAE": depart["mae"].mean(), "R²": depart["r2"].mean(),
                  **dict(zip(ANNEES, depart["rmse"]))}
    synthese_legere[famille] = {"RMSE départ": depart["rmse"].mean(), "RMSE tuning léger": essai["cv_rmse_mean"],
                                "meilleur": "tuning léger" if tuning_meilleur else "départ", **scores,
                                "minutes": round(duree / 60, 1)}

pd.DataFrame.from_dict(synthese_legere, orient="index").sort_values("RMSE").round(3)

,RMSE départ,RMSE tuning léger,meilleur,RMSE,MAE,R²,2008,2009,2010,2011,2012,minutes
extra_trees,1.442,1.447,départ,1.442,0.710,0.971,1.211,1.447,1.385,1.568,1.600,1.2
catboost,1.851,1.458,tuning léger,1.458,0.753,0.970,1.231,1.452,1.387,1.547,1.673,2.4
lightgbm,1.843,1.483,tuning léger,1.483,0.779,0.969,1.284,1.470,1.414,1.571,1.678,0.4
hist_gradient_boosting,1.844,1.497,tuning léger,1.497,0.766,0.968,1.255,1.535,1.408,1.632,1.656,1.0
xgboost,1.693,1.503,tuning léger,1.503,0.784,0.968,1.266,1.544,1.393,1.611,1.702,0.7
random_forest,1.609,1.570,tuning léger,1.570,0.791,0.965,1.387,1.738,1.498,1.657,1.569,1.6


**Observations :**

- Le tuning léger améliore nettement les boostings (−0,19 à −0,39 t/ha), peu RandomForest (−0,04) et pas ExtraTrees :
  ses 40 essais font un peu moins bien que son réglage de départ (1,447 contre 1,442).
- ExtraTrees reste devant (1,442), puis CatBoost (1,458). LightGBM, HistGradientBoosting et XGBoost se suivent en
  0,02 t/ha (1,483 à 1,503) ; RandomForest est nettement derrière (1,570).
- Finalistes : ExtraTrees, CatBoost et LightGBM, les trois meilleures familles. HistGradientBoosting et XGBoost
  restent derrière LightGBM (+0,014 et +0,020 t/ha), RandomForest nettement (+0,09).

# Tuning approfondi des trois finalistes

Chaque grille est centrée sur la meilleure zone du tuning léger (pour ExtraTrees, son réglage de départ) et élargie
d'un cran là où la meilleure valeur touchait un bord. Toutes les combinaisons sont testées, sur les mêmes lignes et
les mêmes folds : 150 pour ExtraTrees, 256 pour CatBoost, 729 pour LightGBM.

In [13]:
FINALISTES = ["extra_trees", "catboost", "lightgbm"]  # les trois meilleures familles du tuning léger
# restent à leur valeur par défaut, la meilleure du tuning léger : min_samples_leaf=1 et bootstrap=False pour
# ExtraTrees, colsample_bytree=1.0 pour LightGBM (subsample_freq=1 est fixé pour tout le tuning)
GRILLES_APPROFONDIES = {
    "extra_trees": {"n_estimators": [200, 300, 400, 500, 600, 800], "max_features": [0.6, 0.7, 0.8, 0.9, 1.0],
                    "min_samples_split": [2, 3, 4, 5, 6]},
    "catboost": {"iterations": [1000, 1500, 2000, 2500], "learning_rate": [0.1, 0.15, 0.2, 0.3], "depth": [10, 11, 12, 13],
                 "l2_leaf_reg": [3, 5, 10, 20]},
    "lightgbm": {"n_estimators": [1000, 1500, 2000], "learning_rate": [0.05, 0.1, 0.15], "num_leaves": [127, 255, 511],
                 "min_child_samples": [10, 20, 40], "subsample": [0.5, 0.6, 0.7], "reg_lambda": [5.0, 10.0, 20.0]},
}
for famille in FINALISTES:
    grille = GRILLES_APPROFONDIES[famille]
    combinaisons = int(np.prod([len(valeurs) for valeurs in grille.values()]))  # toutes testées : 150, 256 et 729
    rechercher(famille, grille, combinaisons, "deep_tuning")

extra_trees              history : 150 configurations en  5.2 min | meilleure RMSE 1.436


catboost                 history : 256 configurations en 80.3 min | meilleure RMSE 1.450


lightgbm                 history : 729 configurations en 18.4 min | meilleure RMSE 1.465


In [14]:
def lire_recherche(famille, phase, representation):
    "Cinq meilleures RMSE, position de la meilleure configuration et plage des meilleures RMSE selon chaque valeur."
    essais, duree, espace = recherches[famille, representation, phase]
    print(f"\n{famille}, {representation} : {len(essais)} configurations en {duree / 60:.0f} min"
          f" | meilleures RMSE {essais['cv_rmse_mean'].head(5).round(3).tolist()}")
    effets = value_effects(essais, espace)["meilleure RMSE"].groupby(level=0)
    plage = effets.agg(lambda serie: f"{serie.min():.3f} à {serie.max():.3f}").rename("meilleure RMSE selon la valeur")
    display(position_in_range(essais, espace).join(plage))


for famille in FINALISTES:
    lire_recherche(famille, "deep_tuning", "history")


extra_trees, history : 150 configurations en 5 min | meilleures RMSE [1.436, 1.437, 1.437, 1.438, 1.439]


,meilleure valeur,valeurs testées,position,meilleure RMSE selon la valeur
hyperparamètre,,,,
n_estimators,300,"[200, 300, 400, 500, 600, 800]",intérieur,1.436 à 1.439
max_features,0.9,"[0.6, 0.7, 0.8, 0.9, 1.0]",intérieur,1.436 à 1.446
min_samples_split,3,"[2, 3, 4, 5, 6]",intérieur,1.436 à 1.482



catboost, history : 256 configurations en 80 min | meilleures RMSE [1.45, 1.45, 1.452, 1.453, 1.454]


,meilleure valeur,valeurs testées,position,meilleure RMSE selon la valeur
hyperparamètre,,,,
iterations,1000,"[1000, 1500, 2000, 2500]",début de plage,1.450 à 1.453
learning_rate,0.2,"[0.1, 0.15, 0.2, 0.3]",intérieur,1.450 à 1.465
depth,12,"[10, 11, 12, 13]",intérieur,1.450 à 1.459
l2_leaf_reg,3,"[3, 5, 10, 20]",début de plage,1.450 à 1.456



lightgbm, history : 729 configurations en 18 min | meilleures RMSE [1.465, 1.465, 1.468, 1.468, 1.469]


,meilleure valeur,valeurs testées,position,meilleure RMSE selon la valeur
hyperparamètre,,,,
n_estimators,1500,"[1000, 1500, 2000]",intérieur,1.465 à 1.469
learning_rate,0.1,"[0.05, 0.1, 0.15]",intérieur,1.465 à 1.477
num_leaves,127,"[127, 255, 511]",début de plage,1.465 à 1.469
min_child_samples,10,"[10, 20, 40]",début de plage,1.465 à 1.476
subsample,0.7,"[0.5, 0.6, 0.7]",fin de plage,1.465 à 1.471
reg_lambda,20.0,"[5.0, 10.0, 20.0]",fin de plage,1.465 à 1.473


**Observations :**

- Les trois grilles sont explorées en entier : 150, 256 et 729 combinaisons. Les cinq meilleures configurations de
  chaque famille tiennent en 0,004 t/ha : chaque sommet est un plateau.
- ExtraTrees garde 300 arbres (1,436 contre 1,442 au départ) : en ajouter n'aide pas (1,436 à 1,439 entre 200 et 800
  arbres). Le gain vient de `max_features=0.9` et `min_samples_split=3`, et aucune meilleure valeur ne touche un bord.
- CatBoost atteint 1,450 avec 1 000 itérations et une profondeur de 12, soit l'une des configurations les plus rapides ;
  2 500 itérations et une profondeur de 13 donnent la même chose. LightGBM atteint 1,465. Pour ces deux familles,
  quelques meilleures valeurs touchent un bord, mais sur ces réglages la RMSE ne varie que de 0,003 à 0,011 t/ha d'un
  bout à l'autre de la plage.

# Comparaison des finalistes

La meilleure configuration de chaque finaliste, réévaluée sur les lignes qui ont un historique. Chaque finaliste est
ensuite entraîné sur toutes ces lignes, sauvegardé, rechargé, puis chronométré sur une demande `/recommend` de 10
cultures.

In [15]:
CANDIDATS = {}
for famille in FINALISTES:
    essais, _, _ = recherches[famille, "history", "deep_tuning"]
    modele, natif, _ = MODELES[famille]
    CANDIDATS[famille] = clone(modele).set_params(**FIXES.get(famille, {}), **essais.iloc[0]["params"])
    print(f"{famille:12s} {essais.iloc[0]['params']}")
    nom = evaluer(famille, CANDIDATS[famille], natif, "history", donnees=MEMES_LIGNES, suffixe=" (réglé)", run=False)
    assert np.allclose(resultats[nom]["rmse"], [essais.iloc[0][f"rmse_{annee}"] for annee in ANNEES])  # scores de l'essai

# une demande /recommend : un pays et ses conditions de 2012, les 10 cultures
categorielles, numeriques, _ = REPRESENTATIONS["history"]
colonnes = categorielles + numeriques
ligne = X_hist[(X_hist["iso3"] == "FRA") & (X_hist["year"] == 2012)].iloc[[0]]
demande = pd.concat([ligne.assign(crop=culture) for culture in RECOMMEND_CROPS], ignore_index=True)

mesures = {}
with tempfile.TemporaryDirectory() as dossier:
    for famille, modele in CANDIDATS.items():
        pipeline = experiment_pipeline(clone(modele), categorielles, numeriques, native_categorical=MODELES[famille][1])
        debut = time.perf_counter()
        pipeline.fit(X_hist[colonnes], y_hist)
        apprentissage = time.perf_counter() - debut
        chemin = Path(dossier) / f"{famille}.joblib"
        joblib.dump(pipeline, chemin)
        recharge = joblib.load(chemin)
        # une forêt additionne ses arbres en parallèle : l'ordre des additions peut changer le 15e chiffre
        assert np.abs(recharge.predict(demande[colonnes]) - pipeline.predict(demande[colonnes])).max() < 1e-9
        debut = time.perf_counter()
        for _ in range(20):
            recharge.predict(demande[colonnes])
        mesures[famille] = {"taille (Mo)": round(chemin.stat().st_size / 1e6, 1), "apprentissage (s)": round(apprentissage, 1),
                            "prédiction de 10 cultures (ms)": round((time.perf_counter() - debut) / 20 * 1000, 1)}

noms = [f"{famille} | history (réglé)" for famille in FINALISTES]
scores = tableau(noms).loc[noms, ["RMSE", "MAE", "R²", *ANNEES]]  # scores des finalistes
scores.index = FINALISTES
scores.join(pd.DataFrame(mesures).T)

extra_trees  {'n_estimators': 300, 'min_samples_split': 3, 'max_features': 0.9}


catboost     {'learning_rate': 0.2, 'l2_leaf_reg': 3, 'iterations': 1000, 'depth': 12}


lightgbm     {'subsample': 0.7, 'reg_lambda': 20.0, 'num_leaves': 127, 'n_estimators': 1500, 'min_child_samples': 10, 'learning_rate': 0.1}


,RMSE,MAE,R²,2008,2009,2010,2011,2012,taille (Mo),apprentissage (s),prédiction de 10 cultures (ms)
extra_trees,1.436,0.713,0.971,1.226,1.459,1.367,1.547,1.581,425.0,0.4,30.3
catboost,1.450,0.740,0.970,1.241,1.412,1.412,1.507,1.678,65.4,7.0,1.0
lightgbm,1.465,0.757,0.970,1.248,1.461,1.410,1.547,1.659,17.5,1.9,2.1


**Observations :**

- ExtraTrees est devant (1,436 t/ha, MAE 0,713) et meilleur 3 années sur 5 ; CatBoost suit (1,450) et gagne en 2009 et
  2011 ; LightGBM ferme la marche (1,465). Les trois tiennent en 0,03 t/ha.
- Leurs coûts restent très différents : 425 Mo pour ExtraTrees, 65 Mo pour CatBoost, 17 Mo pour LightGBM ; une demande
  de 10 cultures est prédite en 30 ms, 1 ms et 2 ms. L'apprentissage va de 0,4 s à 7 s.
- Pour des écarts aussi faibles, le choix ne peut pas se faire sur la seule RMSE : il se fera dans le notebook 15, sur le
  comportement réel de `/recommend`, avant le test final sur 2013.

# Conclusion

In [16]:
journal = pd.DataFrame([
    {"famille": famille, "représentation": representation, "phase": phase, "configurations": len(essais),
     "minutes": round(duree / 60, 1), "meilleure RMSE": round(essais["cv_rmse_mean"].iloc[0], 3)}
    for (famille, representation, phase), (essais, duree, _) in recherches.items()
])
print(f"évaluations hors tuning : {len(resultats)} | configurations de tuning : {journal['configurations'].sum()}"
      f" | durée du notebook : {(time.perf_counter() - debut_notebook) / 60:.0f} min")
journal

évaluations hors tuning : 44 | configurations de tuning : 1375 | durée du notebook : 138 min


,famille,représentation,phase,configurations,minutes,meilleure RMSE
0,xgboost,history,light_tuning,40,0.7,1.503
1,lightgbm,history,light_tuning,40,0.4,1.483
2,hist_gradient_boosting,history,light_tuning,40,1.0,1.497
3,catboost,history,light_tuning,40,2.4,1.458
4,extra_trees,history,light_tuning,40,1.2,1.447
5,random_forest,history,light_tuning,40,1.6,1.570
6,extra_trees,history,deep_tuning,150,5.2,1.436
7,catboost,history,deep_tuning,256,80.3,1.450
8,lightgbm,history,deep_tuning,729,18.4,1.465


- Les modèles à arbres battent largement la régression linéaire de référence (4,673 t/ha).
- La géographie et les conditions historiques améliorent les résultats ; les interactions par culture n'apportent pas de gain régulier.
- Sur `history`, ExtraTrees (1,436), CatBoost (1,450) et LightGBM (1,465) sont très proches, mais leurs tailles diffèrent fortement (425 Mo, 65 Mo, 17 Mo).
- Le modèle final sera choisi dans le notebook 15 selon son comportement réel dans `/recommend`, avant le test final sur 2013.

In [17]:
meilleur = min(FINALISTES, key=lambda famille: resultats[f"{famille} | history (réglé)"]["rmse"].mean())
notify("Agritech /recommend", f"Modèles non linéaires terminés — {meilleur} + history :"
       f" RMSE {resultats[f'{meilleur} | history (réglé)']['rmse'].mean():.3f} t/ha")